# pyceles Notebook: CELES-Style Workflow, CuPy First

This notebook keeps the original CELES-inspired look and feel, but now prefers the CuPy backend automatically when a recent NVIDIA GPU and a working CuPy install are available.

The structure is:

1. **CELES-style benchmark workflow**
   - 500-particle geometry from `examples/sphere_parameters.txt`
   - Gaussian beam, normal incidence
   - GMRES solve with CELES-style mixed precision
   - near field, Poynting overlay, far field, and HDF5 save
2. **Beyond CELES showcase**
   - same benchmark geometry transformed into
     - layered spheres with void cores
     - mildly prolate, randomly rotated spheroids
   - tilted Laguerre-Gaussian illumination
   - near- and far-field visualization on the same compact canvas

If CuPy is unavailable, the notebook falls back to the NumPy reference path automatically.

In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for cand in (start, *start.parents):
        if (
            (cand / 'pyproject.toml').exists()
            and (cand / 'src' / 'pyceles').exists()
            and (cand / 'examples' / 'sphere_parameters.txt').exists()
        ):
            return cand
    raise FileNotFoundError(
        'Could not find repository root containing src/pyceles and examples/sphere_parameters.txt.'
    )


_repo_root = _find_repo_root(Path.cwd().resolve())
_src_path = _repo_root / 'src'
if str(_src_path) not in sys.path:
    sys.path.insert(0, str(_src_path))

import matplotlib.pyplot as plt
import numpy as np
import pyceles as pcl

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 180


def choose_backend() -> tuple[str, str, str | None]:
    try:
        import cupy as cp

        ndev = int(cp.cuda.runtime.getDeviceCount())
        if ndev > 0:
            name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode(errors='ignore')
            return 'cupy', 'inherit', name
    except Exception as exc:  # graceful fallback for no CuPy / no GPU / broken CUDA
        print(f'CuPy unavailable, falling back to NumPy: {exc}')
    return 'numpy', 'numpy', None


operator_backend, postprocessing_backend, gpu_name = choose_backend()
print(f'Operator backend: {operator_backend}')
print(f'Postprocessing backend: {postprocessing_backend}')
if gpu_name is not None:
    print(f'CuPy device: {gpu_name}')


## 1) CELES-Style 500-Particle Workflow

This section mirrors the classic CELES benchmark setup, but with the current pyceles API.

Notes:
- the notebook prefers `operator_backend='cupy'` automatically;
- both NumPy and CuPy runs use the matrix-free solver without a built-in pyceles preconditioner;
- CuPy postprocessing follows the solve backend by default when accelerated kernels exist.

In [ ]:
sphere_data = np.loadtxt(_repo_root / 'examples' / 'sphere_parameters.txt')
benchmark_n_particles = 500

radii = sphere_data[:benchmark_n_particles, 0].astype(float)
positions = sphere_data[:benchmark_n_particles, 1:4].astype(float)
refractive_indices = (
    sphere_data[:benchmark_n_particles, 4] + 1j * sphere_data[:benchmark_n_particles, 5]
).astype(np.complex128)
particles = pcl.spheres_from_arrays(
    positions=positions,
    radii=radii,
    refractive_indices=refractive_indices,
)

source = pcl.GaussianBeam(
    wavelength=550.0,
    medium_n=1.0 + 0j,
    polarization='TE',
    polar_angle=0.0,
    azimuthal_angle=0.0,
    beam_width=2000.0,
    focal_point=(0.0, 0.0, 0.0),
    amplitude=1.0,
)

cfg = pcl.SimulationConfig(
    wavelength=550.0,
    n_medium=1.0 + 0j,
    lmax=3,
    polar_angles=pcl.core.uniform_polar_grid(721),
    azimuthal_angles=pcl.core.uniform_periodic_azimuth_grid(360),
    solver_method='gmres',
    solver_rtol=1e-4,
    solver_restart=25,
    solver_maxiter=100,
    compute_dtype='complex64',
    accum_dtype='complex128',
    operator_backend=operator_backend,
    postprocessing_backend=postprocessing_backend,
    verbose=True,
)

sim = pcl.Simulation(cfg, particles=particles)


In [ ]:
run = sim.run(source)
res = run.solver_result

print('Solver method:', res.method)
print('Solver info:', res.info)
print('Solver iterations:', res.iterations)
print('Relative residual:', res.relative_residual)
print('||x||:', np.linalg.norm(res.x))

if run.power is not None:
    P0 = run.power.incident_power
    Pf = run.power.transmitted_power
    Pb = run.power.reflected_power
    closure_raw = run.power.flux_defect_fraction

    print(f'initial power:     {P0:.6e}')
    print(f'T (transmitted):   {Pf / P0 * 100:.4f} %')
    print(f'R (reflected):     {Pb / P0 * 100:.4f} %')
    print(f'1 - T - R (raw):   {closure_raw * 100:.4f} %')

    print(f'local absorbed:    {run.power.local_absorbed_power:.6e}')
    print(f'closure error:     {run.power.closure_error:.6e}')
    print(f'local absorptance:  {run.power.local_absorptance:.6f}')

if run.cross_sections is not None:
    cs = run.cross_sections
    print('Cross sections:')
    print(
        f'  extinction: {cs.extinction:.6e}  '
        f'scattering: {cs.scattering:.6e}  '
        f'local_absorption: {cs.local_absorption:.6e}'
    )
    print(
        f'  absorption_by_difference: {cs.absorption_by_difference:.6e}  '
        f'closure_error: {cs.closure_error:.6e}'
    )


In [ ]:
nf_slice = pcl.compute_near_field_slice(
    run,
    axis_0_min=-4000.0,
    axis_0_max=4000.0,
    axis_1_min=-3000.0,
    axis_1_max=5000.0,
    dx=40.0,
    plane='y',
    plane_value=0.0,
    show_progress=True,
)

E, H = nf_slice.field_maps['total']
fig, _ = pcl.io.plot_nearfield_panels(
    nf_slice.axis_0,
    nf_slice.axis_1,
    E,
    H,
    particles=run.particles,
    plane=nf_slice.plane,
    plane_value=nf_slice.plane_value,
    real_limits=(-2.0, 2.0),
    abs_limits=(0.0, 2.0),
)
plt.show()

fig, _ = pcl.io.plot_nearfield_poynting_overlay(
    nf_slice.axis_0,
    nf_slice.axis_1,
    E,
    H,
    particles=run.particles,
    plane=nf_slice.plane,
    plane_value=nf_slice.plane_value,
    stride=5,
    quiver_color='w',
)
plt.show()


In [ ]:
ff = run.farfield
if ff.total_te is not None and ff.total_tm is not None:
    pwp_te, pwp_tm = ff.total_te, ff.total_tm
else:
    pwp_te, pwp_tm = ff.scattered_te, ff.scattered_tm

Iff = pcl.io.far_field_intensity(pwp_te, pwp_tm)
fig, _ = pcl.io.plot_farfield_hemispheres(
    polar_angles=pwp_te['beta'],
    azimuthal_angles=pwp_te['alpha'],
    intensity=Iff,
    cmap='inferno',
    independent_scales=True,
)
plt.show()

out_h5 = _repo_root / 'outputs' / '01_pyceles_main_replication.h5'
out_h5.parent.mkdir(parents=True, exist_ok=True)
pcl.save_simulation_h5(run, nf_slice, out_h5)
print('Saved:', out_h5.resolve())


## 2) Beyond CELES: New Particle Families and Sources

This section keeps the original benchmark centers and circumscribing radii, but transforms the particle family:
- **Layered-sphere cluster**: vacuum-like core with the original outer shell as the high-index boundary;
- **Spheroid cluster**: mildly prolate particles with random Euler angles.

To keep notebook runtime reasonable on a NumPy fallback, the showcase uses fewer particles by default there. On CuPy it uses the full `500`-particle transformed cluster.

The source is now a tilted Laguerre-Gaussian beam with a Jones-polarized TE/TM mixture.

In [ ]:
rng = np.random.default_rng(7)
showcase_n_particles = 500 if operator_backend == 'cupy' else 160
showcase_radii = radii[:showcase_n_particles]
showcase_positions = positions[:showcase_n_particles]
showcase_n = refractive_indices[:showcase_n_particles]


def make_layered_cluster() -> list[pcl.LayeredSphere]:
    layered: list[pcl.LayeredSphere] = []
    for pos, r, n_shell in zip(showcase_positions, showcase_radii, showcase_n, strict=True):
        core_radius = 0.6 * float(r)
        layered.append(
            pcl.LayeredSphere(
                position=tuple(map(float, pos)),
                layer_radii=(core_radius, float(r)),
                layer_refractive_indices=(1.0 + 0.0j, complex(n_shell)),
            )
        )
    return layered


def make_spheroid_cluster() -> list[pcl.Spheroid]:
    spheroids: list[pcl.Spheroid] = []
    for pos, r, n_part in zip(showcase_positions, showcase_radii, showcase_n, strict=True):
        spheroids.append(
            pcl.Spheroid(
                position=tuple(map(float, pos)),
                equatorial_radius=0.6 * float(r),
                polar_radius=float(r),
                refractive_index=complex(n_part),
                euler_angles=tuple(map(float, rng.uniform(0.0, 2.0 * np.pi, size=3))),
            )
        )
    return spheroids


source_showcase = pcl.LaguerreGaussianBeam(
    wavelength=550.0,
    medium_n=1.0 + 0j,
    polarization=((1.0 + 0j) / np.sqrt(2.0), 1.0j / np.sqrt(2.0)),
    radial_order_p=0,
    azimuthal_order_l=1,
    polar_angle=0.32,
    azimuthal_angle=0.21,
    beam_width=1800.0,
    focal_point=(0.0, 0.0, 0.0),
    amplitude=1.0,
)

cfg_showcase = pcl.SimulationConfig(
    wavelength=550.0,
    n_medium=1.0 + 0j,
    lmax=3,
    polar_angles=pcl.core.uniform_polar_grid(721),
    azimuthal_angles=pcl.core.uniform_periodic_azimuth_grid(360),
    solver_method='gmres',
    solver_rtol=1e-4,
    solver_restart=25,
    solver_maxiter=100,
    compute_dtype='complex64',
    accum_dtype='complex128',
    operator_backend=operator_backend,
    postprocessing_backend=postprocessing_backend,
    verbose=True,
)

layered_particles = make_layered_cluster()
spheroid_particles = make_spheroid_cluster()
print(f'Showcase particles: {showcase_n_particles}')


In [ ]:
def run_showcase_case(label: str, particles_case: list[pcl.core.Particle]) -> tuple[pcl.SimulationResult, pcl.postprocessing.NearFieldSlice]:
    sim_case = pcl.Simulation(cfg_showcase, particles=particles_case)
    run_case = sim_case.run(source_showcase)
    nf_case = pcl.compute_near_field_slice(
        run_case,
        axis_0_min=-1500.0,
        axis_0_max=1500.0,
        axis_1_min=0.0,
        axis_1_max=3000.0,
        dx=30.0,
        plane='y',
        plane_value=0.0,
        show_progress=True,
    )

    ff_case = run_case.farfield
    intensity_case = pcl.io.far_field_intensity(ff_case.scattered_te, ff_case.scattered_tm)

    print(f'[{label}] iterations =', run_case.solver_result.iterations)
    print(f'[{label}] residual =', run_case.solver_result.relative_residual)
    P0 = run_case.power.incident_power
    T = run_case.power.transmittance
    R = run_case.power.reflectance
    print(f'[{label}] T = {T:.6f}, R = {R:.6f}, flux defect = {run_case.power.flux_defect_fraction:.6f}')
    print(f'[{label}] local absorptance = {run_case.power.local_absorptance:.6f}, closure = {run_case.power.closure_error_fraction:.6e}')

    fig_nf, _ = pcl.io.plot_nearfield_panels(
        nf_case.axis_0,
        nf_case.axis_1,
        nf_case.field_maps['total'][0],
        nf_case.field_maps['total'][1],
        particles=run_case.particles,
        plane=nf_case.plane,
        plane_value=nf_case.plane_value,
    )
    fig_nf.suptitle(f'{label}: total near field', y=1.02)
    plt.show()

    fig_ff, _ = pcl.io.plot_farfield_hemispheres(
        polar_angles=ff_case.scattered_te['beta'],
        azimuthal_angles=ff_case.scattered_te['alpha'],
        intensity=intensity_case,
        cmap='magma',
        independent_scales=True,
        title=f'{label}: scattered far-field intensity',
    )
    plt.show()

    return run_case, nf_case


run_layered, nf_layered = run_showcase_case('Layered-sphere cluster', layered_particles)
run_spheroid, nf_spheroid = run_showcase_case('Rotated spheroid cluster', spheroid_particles)


### Optional LDOS / Purcell-like `p_x` map on the CELES-style cluster

This section revisits the old notebook LDOS idea, but now prefers the **direct CuPy solve** when the CuPy backend is active.

Design choices:
- only **x-oriented dipoles** (`p_x`) are sampled, not the orientation average;
- the physical canvas is `x in [-1500, 1500] nm`, `z in [0, 3000] nm`, `dx = 30 nm`, i.e. a `101 x 101` map;
- internal points are masked out before solving;
- a **pilot solve** is always run first to estimate runtime;
- the full map is **off by default** to avoid launching thousands of repeated solves accidentally.

On the CuPy path this section now uses the cached dense direct solve, which is a much better fit for repeated LDOS probe solves than restarted GMRES.

In [ ]:
from time import perf_counter
from tqdm.auto import tqdm

ldos_run_full_map = False
ldos_stride = 1
ldos_max_pixels = None
ldos_dx = 30.0
ldos_x_min, ldos_x_max = -1500.0, 1500.0
ldos_z_min, ldos_z_max = 0.0, 3000.0
ldos_moment_magnitude = 1.0 + 0j

x_ldos = np.arange(ldos_x_min, ldos_x_max + 0.5 * ldos_dx, ldos_dx, dtype=float)
z_ldos = np.arange(ldos_z_min, ldos_z_max + 0.5 * ldos_dx, ldos_dx, dtype=float)
xx_ldos, zz_ldos = np.meshgrid(x_ldos, z_ldos)
points_ldos = np.stack([xx_ldos, np.zeros_like(xx_ldos), zz_ldos], axis=-1)
points_flat = points_ldos.reshape(-1, 3)

inside = np.zeros((points_flat.shape[0],), dtype=bool)
if run.positions.shape[0] > 0:
    dr = points_flat[:, None, :] - run.positions[None, :, :]
    inside = np.any(np.linalg.norm(dr, axis=2) < run.circumscribing_radii[None, :], axis=1)
inside = inside.reshape(xx_ldos.shape)

valid = ~inside
row_ids, col_ids = np.indices(valid.shape)
sampled = valid & (row_ids % int(ldos_stride) == 0) & (col_ids % int(ldos_stride) == 0)
pixel_ids = np.argwhere(sampled)
if ldos_max_pixels is not None:
    pixel_ids = pixel_ids[: int(ldos_max_pixels)]

print(f'LDOS valid pixels: {int(np.count_nonzero(valid))}')
print(f'LDOS sampled pixels (stride={ldos_stride}): {int(pixel_ids.shape[0])}')
if pixel_ids.shape[0] == 0:
    raise RuntimeError('No sampled LDOS pixels. Reduce stride or check the canvas.')

probe_template = pcl.DipoleSource(
    wavelength=float(run.config.wavelength),
    medium_n=complex(run.config.n_medium),
    position=(0.0, 0.0, 0.0),
    dipole_moment=(ldos_moment_magnitude, 0.0 + 0j, 0.0 + 0j),
)

# LDOS probes are repeated-RHS solves on a fixed geometry, so both backends benefit
# from the direct solver + cached LU path when the dense operator still fits.
ldos_solver_method = 'direct'

cfg_ldos = pcl.SimulationConfig(
    wavelength=float(run.config.wavelength),
    n_medium=complex(run.config.n_medium),
    lmax=3,
    polar_angles=run.config.polar_angles,
    azimuthal_angles=run.config.azimuthal_angles,
    solver_method=ldos_solver_method,
    solver_direct_max_n=max(15000, run.coeffs.shape[0] * run.coeffs.shape[1]),
    solver_rtol=1e-4,
    solver_restart=25,
    solver_maxiter=100,
    compute_dtype=run.config.compute_dtype,
    accum_dtype=run.config.accum_dtype,
    operator_backend=operator_backend,
    postprocessing_backend=postprocessing_backend,
    verbose=False,
)

sim_ldos = pcl.Simulation(cfg_ldos, particles=particles)
print(f'LDOS solver method: {cfg_ldos.solver_method}')


def _solve_px_ldos(point_xyz: np.ndarray) -> float:
    dip = pcl.DipoleSource(
        wavelength=float(run.config.wavelength),
        medium_n=complex(run.config.n_medium),
        position=tuple(map(float, point_xyz)),
        dipole_moment=(ldos_moment_magnitude, 0.0 + 0j, 0.0 + 0j),
    )
    solved = sim_ldos.solve_sources({'px': dip}, solver_compute_final_residual=False)
    multi = sim_ldos.postprocess_sources(solved, include_farfield=False)
    return float(
        pcl.compute_dipole_ldos_enhancement(
            multi['px'],
            allow_inside_particle=False,
            show_progress=False,
        )
    )

pilot_i, pilot_j = pixel_ids[0]
pilot_xyz = np.asarray([xx_ldos[int(pilot_i), int(pilot_j)], 0.0, zz_ldos[int(pilot_i), int(pilot_j)]], dtype=float)
t0 = perf_counter()
pilot_ldos = _solve_px_ldos(pilot_xyz)
pilot_dt = perf_counter() - t0
print(f'Pilot p_x enhancement: {pilot_ldos:.6f}')

if cfg_ldos.solver_method == 'direct':
    print(f'First direct solve (build + factorize + solve): {pilot_dt:.2f} s')
    t1 = perf_counter()
    pilot_ldos_repeat = _solve_px_ldos(pilot_xyz)
    repeat_dt = perf_counter() - t1
    eta_s = repeat_dt * int(pixel_ids.shape[0])
    print(f'Repeated direct solve (cached LU): {repeat_dt:.2f} s/pixel')
    print(f'Repeated p_x enhancement: {pilot_ldos_repeat:.6f}')
    print(f'Estimated sampled-map runtime (cached LU): {eta_s/60:.1f} min ({eta_s/3600:.2f} h)')
else:
    eta_s = pilot_dt * int(pixel_ids.shape[0])
    print(f'Pilot runtime: {pilot_dt:.2f} s/pixel')
    print(f'Estimated sampled-map runtime: {eta_s/60:.1f} min ({eta_s/3600:.2f} h)')

ldos_px = np.full(xx_ldos.shape, np.nan, dtype=float)
if ldos_run_full_map:
    for i, j in tqdm(pixel_ids, total=int(pixel_ids.shape[0]), desc='LDOS p_x pixels', unit='px'):
        xyz = np.asarray([xx_ldos[int(i), int(j)], 0.0, zz_ldos[int(i), int(j)]], dtype=float)
        ldos_px[int(i), int(j)] = _solve_px_ldos(xyz)

    finite_vals = ldos_px[np.isfinite(ldos_px)]
    if finite_vals.size > 0:
        vmin = float(np.nanpercentile(finite_vals, 2.0))
        vmax = float(np.nanpercentile(finite_vals, 98.0))
    else:
        vmin, vmax = 1.0, 1.0

    fig, ax = plt.subplots(figsize=(7.8, 4.4))
    im = ax.pcolormesh(xx_ldos, zz_ldos, np.ma.masked_invalid(ldos_px), shading='auto', cmap='magma', vmin=vmin, vmax=vmax)
    for p in particles:
        p.plot(ax=ax, plane='xz', facecolor='none', edgecolor='white', linewidth=1.1)
    ax.set_xlabel('x [nm]')
    ax.set_ylabel('z [nm]')
    ax.set_title('LDOS enhancement map for $p_x$ dipoles')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('LDOS enhancement')
    plt.show()
else:
    print('Full LDOS map skipped (set ldos_run_full_map=True to compute sampled pixels).')
